# Módulo 8 · Espectro y tiempo-frecuencia

**Objetivo:** describir cada ventana por su contenido en frecuencia (potencia por bandas = las
*features* clásicas de ML en EEG) y crear **espectrogramas** — una de las visualizaciones de
alto impacto para tu memoria, y también una posible entrada para redes 2D (CNN sobre espectrogramas).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, preprocess, make_windows, BANDS

mne.set_log_level("WARNING")
raw = preprocess(read_tusz_raw("aaaaabqv_s001_t000"))
ep = make_windows(raw, win=4.0, step=2.0)
ep

## 8.1 PSD por epoch: `epochs.compute_psd()` → `EpochsSpectrum`

In [ ]:
spec = ep.compute_psd(method="welch", fmin=0.5, fmax=40, n_fft=400, n_overlap=200)
psd, freqs = spec.get_data(return_freqs=True)
print(psd.shape, "→ (ventanas, canales, frecuencias)", "| resolución:", freqs[1] - freqs[0], "Hz")

El objeto `Spectrum` también se indexa con metadata, como las epochs:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lab, c in (("interictal", "tab:gray"), ("ictal", "tab:red")):
    p = spec[f'label == "{lab}"'].get_data().mean(1)       # media de canales → (n, freqs)
    m, sd = np.log10(p).mean(0), np.log10(p).std(0)
    ax.plot(freqs, m, c=c, label=f"{lab} (n={len(p)})")
    ax.fill_between(freqs, m - sd, m + sd, color=c, alpha=0.2)
for name, (lo, hi) in BANDS.items():
    ax.axvspan(lo, hi, alpha=0.04, color="k"); ax.text((lo + hi) / 2, ax.get_ylim()[1], name, ha="center", va="top", fontsize=8)
ax.set_xlabel("Hz"); ax.set_ylabel("log10 PSD (V²/Hz)"); ax.legend(); ax.set_title("Espectro ictal vs. interictal")
plt.show()

## 8.2 Potencia por bandas (features clásicas)

Para cada ventana y canal: potencia en δ (0.5-4), θ (4-8), α (8-13), β (13-30), γ (30-40).
Suele usarse **relativa** (dividida por la total) y en **log**.

In [ ]:
from tusz_utils import band_power
bp = band_power(ep, relative=True, log=False)
print(bp.shape, "→ (ventanas, 22 canales, 5 bandas)")

df = pd.DataFrame(bp.mean(1), columns=list(BANDS))
df["label"] = ep.metadata.label.values
df.groupby("label").mean().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df.melt(id_vars="label", var_name="banda", value_name="potencia relativa") \
  .pivot_table(index="banda", columns="label", values="potencia relativa").loc[list(BANDS)].plot.bar(ax=ax, rot=0)
ax.set_title("Potencia relativa por banda"); plt.show()

En la ausencia sube **δ** (la punta-onda a 3 Hz cae justo ahí) y baja α/β. Una regresión logística con
estas 22 × 5 = 110 features ya separa bastante bien (lo verás en el módulo 9).

## 8.3 Tiempo-frecuencia con Morlet: `compute_tfr`

El PSD pierde el tiempo. Las **wavelets de Morlet** dan potencia en cada (frecuencia, instante).
Lo aplicamos a epochs largas centradas en el inicio de cada crisis:

In [ ]:
events, eid = mne.events_from_annotations(raw)
ep_on = mne.Epochs(raw, events, eid, tmin=-10, tmax=15, baseline=None, preload=True)
freqs_tfr = np.arange(1, 31, 0.5)
tfr = ep_on.compute_tfr(method="morlet", freqs=freqs_tfr, n_cycles=freqs_tfr / 2, decim=4,
                        average=True, return_itc=False)
tfr

In [ ]:
fig = tfr.plot(picks=["FP2-F4"], baseline=(-10, -1), mode="logratio", title="FP2-F4 · log(potencia / basal)",
               cmap="RdBu_r", show=False)
plt.show()

En t = 0 la potencia sube **de golpe en todas las frecuencias** y dura ~7 s: es la crisis.
Fíjate en las **rayas verticales regulares** (~3 por segundo): cada una es una *punta* de la punta-onda.
Una punta es un evento muy rápido, así que reparte energía por todo el espectro (incluidas las altas
frecuencias); por eso en el módulo 6 el detector de "músculo" se disparaba en las crisis. La onda lenta
a 3 Hz y sus armónicos (6, 9 Hz…) se ven mejor en el PSD de 8.1 que aquí.

- `n_cycles`: compromiso resolución temporal ↔ frecuencial (más ciclos = más precisión en frecuencia).
- `baseline` + `mode="logratio"`: normaliza cada frecuencia por su potencia basal; imprescindible porque
  el EEG tiene mucha más potencia en bajas frecuencias (ley 1/f).
- `decim`: reduce el muestreo del resultado para ahorrar memoria.

In [ ]:
# Media de los 22 canales (los canales bipolares no tienen posición en la cabeza,
# así que aquí no hay plot_topo; con datos referenciales + montaje 10-20 sí lo habría)
fig = tfr.plot(picks="eeg", combine="mean", baseline=(-10, -1), mode="logratio",
               title="Media de canales TCP", cmap="RdBu_r", show=False)
plt.show()

## 8.4 Espectrograma del registro completo con las crisis superpuestas

Una figura "de portada" para la memoria: todo el registro, potencia en el tiempo y las anotaciones encima.
Para registros largos es más rápido `scipy.signal.spectrogram` sobre el canal (o la media):

In [ ]:
from scipy.signal import spectrogram
x = raw.get_data(units="uV")
f, t, S = spectrogram(x, fs=raw.info["sfreq"], nperseg=400, noverlap=300)
S = np.log10(S.mean(0) + 1e-12)                     # media de canales, log
keep = (f >= 0.5) & (f <= 30)
f, S = f[keep], S[keep]
# Normalizamos cada frecuencia por su mediana en el registro: así resalta lo que cambia
# respecto a lo "habitual" de ese paciente (si no, domina la ley 1/f y los tramos planos).
S_rel = S - np.median(S, axis=1, keepdims=True)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(13, 5), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
im = ax0.pcolormesh(t, f, S_rel, shading="auto", cmap="magma", vmin=-0.5, vmax=1.5)
ax0.set_ylabel("Hz"); ax0.set_title("aaaaabqv_s001_t000 · espectrograma relativo (media 22 canales TCP)")
for a in raw.annotations:
    for ax in (ax0, ax1):
        ax.axvspan(a["onset"], a["onset"] + a["duration"], ec="cyan", fc="none", lw=1.2)
ax1.plot(t, S_rel[(f >= 2) & (f <= 4)].mean(0), lw=0.7, c="k")
ax1.set_ylim(-1, 2); ax1.set_ylabel("Δ log P\n2-4 Hz"); ax1.set_xlabel("s")
fig.colorbar(im, ax=[ax0, ax1], label="log10 P − mediana", pad=0.01)
plt.show()

In [ ]:
# Zoom a la zona de las crisis
fig, ax = plt.subplots(figsize=(13, 3.5))
z = (t > 150) & (t < 320)
im = ax.pcolormesh(t[z], f, S_rel[:, z], shading="auto", cmap="magma", vmin=-0.5, vmax=1.5)
for a in raw.annotations:
    ax.axvspan(a["onset"], a["onset"] + a["duration"], ec="cyan", fc="none", lw=1.5)
ax.set_ylabel("Hz"); ax.set_xlabel("s"); ax.set_title("Zoom: las 3 ausencias"); fig.colorbar(im, ax=ax)
plt.show()

Esta figura dice mucho:
- Las **líneas horizontales dentro de cada crisis** son la fundamental (~3-4 Hz) y sus **armónicos**:
  la punta-onda no es una sinusoide, así que su energía aparece en múltiplos de su frecuencia.
- Las líneas **se curvan hacia abajo**: la frecuencia de la descarga **disminuye** a lo largo de la crisis
  (de ~4 a ~3 Hz), algo típico de las ausencias.
- En el espectrograma completo hay otras rayas verticales brillantes (≈ 420, 500, 580, 630 s) que **no son
  crisis**: son los artefactos musculares del módulo 6. Serán los candidatos a **falsos positivos** de tu modelo.

> En los módulos de XAI podrás superponer aquí la **probabilidad de crisis predicha por tu red** en la
> fila de abajo: es la visualización "espectrograma + predicciones" que querías para el TFM.

## 8.5 Multitaper (bonus)
Más robusto que Welch para ventanas cortas (menos varianza). Disponible tanto en PSD como en TFR:

In [ ]:
spec_mt = ep.compute_psd(method="multitaper", fmin=0.5, fmax=40, bandwidth=1.0)
print(spec_mt.get_data().shape)

---
## 🧪 Ejercicios

**E8.1** Calcula el **cociente θ/α** y **(δ+θ)/(α+β)** por ventana (media de canales). Dibuja su
evolución temporal con las crisis sombreadas. ¿Son buenos detectores de ausencia?

**E8.2** Repite el TFR de 8.3 con `n_cycles = 3` (fijo) y con `n_cycles = freqs` . ¿Qué cambia?

**E8.3** Crea, para cada ventana de 4 s, un espectrograma pequeño (canales × frecuencias × tiempo) con
`ep.compute_tfr(..., average=False)`. ¿Qué forma tiene el array? Esa sería la entrada de una CNN 2D/3D.

### Solución E8.1

In [ ]:
bpa = band_power(ep, relative=False, log=False).mean(1)          # (n, 5)
d, th, a, b, g = bpa.T
md = ep.metadata
fig, ax = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
ax[0].plot(md.start, th / a, lw=0.7); ax[0].set_ylabel("θ/α")
ax[1].plot(md.start, (d + th) / (a + b), lw=0.7); ax[1].set_ylabel("(δ+θ)/(α+β)"); ax[1].set_xlabel("s")
for x0 in md[md.label == "ictal"].start:
    for axx in ax:
        axx.axvspan(x0, x0 + 4, color="tab:red", alpha=0.03)
plt.show()

### Solución E8.2

In [ ]:
for nc, name in ((3, "n_cycles=3"), (freqs_tfr, "n_cycles=freqs")):
    tf = ep_on.compute_tfr("morlet", freqs=freqs_tfr, n_cycles=nc, decim=4, average=True)
    tf.plot(picks=["FP2-F4"], baseline=(-10, -1), mode="logratio", title=name, cmap="RdBu_r", show=False)
plt.show()
# Pocos ciclos: buena resolución temporal, frecuencias "borrosas". Muchos: al revés.

### Solución E8.3

In [ ]:
ff = np.arange(1, 31, 1.0)
tfr_ep = ep[:50].compute_tfr("morlet", freqs=ff, n_cycles=ff / 2, decim=10, average=False)
print(tfr_ep.get_data().shape, "→ (ventanas, canales, frecuencias, tiempos)")